In [ ]:
%pip install dotenv openpyxl xlsxwriter

In [14]:
import io
import os
import pandas as pd
from dotenv import load_dotenv

# ==========================================
# 1. CONFIGURACION INICIAL
# ==========================================
ENV_FILE = os.getenv("RECON_ENV_FILE", "env.txt")
env_path = ENV_FILE if os.path.isabs(ENV_FILE) else os.path.join(os.getcwd(), ENV_FILE)
load_dotenv(env_path, override=False)

BASE_DIR = os.getenv("RECON_ROOT", "")
OUTPUT_BASE_DIR = os.getenv("RECON_OUTPUT_DIR", "Automatic reconciliation")
USE_BANK_STATEMENTS = False
RECON_IS_S3 = os.getenv("RECON_IS_S3", "false")
IS_S3 = RECON_IS_S3.lower() in {"1", "true", "yes"}
S3FS = None
if IS_S3:
    import s3fs

    S3FS = s3fs.S3FileSystem()

CONFIGS = [
    {
        "sap_file": "SAP Exports/NOV 2025.xlsx",
        "bank_file": "EB_11137280.xlsx",
        "account": "11137280",
        "month_name": "November",
        "month_number": "011",
        "year": "2025",
    },
    {
        "sap_file": "SAP Exports/DEC 2025.xlsx",
        "bank_file": "EB_11137280.xlsx",
        "account": "11137280",
        "month_name": "December",
        "month_number": "012",
        "year": "2025",
    },
    {
        "sap_file": "SAP Exports/JAN 2026.xlsx",
        "bank_file": "EB_11137280.xlsx",
        "account": "11137280",
        "month_name": "January",
        "month_number": "001",
        "year": "2026",
    },
    {
        "sap_file": "SAP Exports/FEB 2026.xlsx",
        "bank_file": "EB_11137280.xlsx",
        "account": "11137280",
        "month_name": "February",
        "month_number": "002",
        "year": "2026",
    },
]

COLUMNAS_SALIDA = [
    'G/L Account', 'Journal Entry', 'Company Code', 'Journal Entry Type', 'Posting Date', 
    'Amount in Company Code Currency', 'Journal Entry Item Text', 'Description',
    'Value date', 'Amount in Transaction Currency', 
    'Offsetting Account', 'Amount in Global Currency', 'Assignment Reference', 
    'Clearing Date', 'Journal Entry Item', 'Fiscal Period', 'Clearing Journal Entry',
    'G/L Account Long Name', 'G/L Account Name',
]

reglas_entidades = {
    'Hoja 1': { 
        'prefix': 'Lib Diff ', 
        'xq_ba': [r'BA.?0600',], 
        'bank_text': [r'NONREF', r'FLOW', r'JAMAICA', r'FCIB FUNDS'] 
    }
}

# ==========================================
# 2. PROCESAMIENTO
# ==========================================

def join_path(base, *parts):
    if IS_S3:
        base_clean = base.rstrip("/")
        tail = "/".join(p.strip("/") for p in parts if p is not None)
        return f"{base_clean}/{tail}" if tail else base_clean
    if base:
        return os.path.join(base, *parts)
    return os.path.join(*parts)


def path_exists(path):
    return S3FS.exists(path) if IS_S3 else os.path.exists(path)


def read_excel_path(path):
    if IS_S3:
        with S3FS.open(path, "rb") as f:
            return pd.read_excel(f)
    return pd.read_excel(path)


def write_excel_to_path(buffer, path):
    if IS_S3:
        with S3FS.open(path, "wb") as f:
            f.write(buffer.getvalue())


def clean_amount(val):
    if pd.isna(val): return 0.0
    if isinstance(val, (int, float)): return float(val)
    val = str(val).replace('JMD', '').replace('USD', '').replace(',', '').strip()
    try: return float(val)
    except: return 0.0


def build_desc_lookup(df_bank):
    desc_col = 'DESCRIPTION'
    debits_col = 'DEBITS'
    credits_col = 'CREDITS'
    if desc_col not in df_bank.columns or (debits_col not in df_bank.columns and credits_col not in df_bank.columns):
        return pd.Series(dtype='object')
    parts = []
    if debits_col in df_bank.columns:
        deb = df_bank[[debits_col, desc_col]].copy()
        deb['amount_match'] = deb[debits_col].apply(clean_amount).abs()
        parts.append(deb[['amount_match', desc_col]])
    if credits_col in df_bank.columns:
        cred = df_bank[[credits_col, desc_col]].copy()
        cred['amount_match'] = cred[credits_col].apply(clean_amount).abs()
        parts.append(cred[['amount_match', desc_col]])
    if not parts:
        return pd.Series(dtype='object')
    eb_long = pd.concat(parts, ignore_index=True)
    eb_long = eb_long[eb_long['amount_match'] != 0]
    eb_long = eb_long.dropna(subset=[desc_col])
    return eb_long.groupby('amount_match')[desc_col].first()


# Formato de fecha Dia/Mes/Ano con barras
def clean_date(val):
    if pd.isna(val) or val == "" or str(val).lower() == "nan": return ""
    try:
        return pd.to_datetime(val).strftime('%d/%m/%Y')
    except:
        return str(val)


for idx, config in enumerate(CONFIGS, start=1):
    ARCHIVO_SAP = join_path(BASE_DIR, config["sap_file"])
    CUENTA_OBJETIVO = config["account"]
    MES_NOMBRE = config["month_name"]
    MES_NUMERO = config["month_number"]
    ANIO = config["year"]
    month_folder = f"{MES_NOMBRE[:3].upper()} {ANIO}"
    ARCHIVO_BANCARIO = join_path(
        BASE_DIR, "Bank Statements", month_folder, config["bank_file"]
    )
    OUTPUT_DIR = join_path(OUTPUT_BASE_DIR, CUENTA_OBJETIVO)
    if not IS_S3:
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not path_exists(ARCHIVO_SAP):
        raise FileNotFoundError(
            f"Iteracion {idx} ({MES_NOMBRE} {ANIO}) - SAP no encontrado: {ARCHIVO_SAP}"
        )
    if USE_BANK_STATEMENTS and not path_exists(ARCHIVO_BANCARIO):
        raise FileNotFoundError(
            f"Iteracion {idx} ({MES_NOMBRE} {ANIO}) - Banco no encontrado: {ARCHIVO_BANCARIO}"
        )

    df_original = read_excel_path(ARCHIVO_SAP)
    eb = read_excel_path(ARCHIVO_BANCARIO) if USE_BANK_STATEMENTS else None

    # Limpieza de montos
    cols_monto = ['Amount in Transaction Currency', 'Amount in Global Currency', 'Amount in Company Code Currency']
    for col in cols_monto:
        if col in df_original.columns:
            df_original[col] = df_original[col].apply(clean_amount)

    # Match por monto absoluto contra Debits/Credits del extracto
    if USE_BANK_STATEMENTS and eb is not None:
        desc_lookup = build_desc_lookup(eb)
        df_original['Description'] = df_original['Amount in Transaction Currency'].abs().map(desc_lookup).fillna("")
    else:
        df_original['Description'] = ""

    df_acc = df_original[df_original['G/L Account'].astype(str).str.contains(CUENTA_OBJETIVO, na=False)].copy()

    output_file = join_path(OUTPUT_DIR, f"{CUENTA_OBJETIVO}_{MES_NOMBRE}_{ANIO}.xlsx")
    if IS_S3:
        output_buffer = io.BytesIO()
        writer = pd.ExcelWriter(output_buffer, engine='xlsxwriter')
    else:
        writer = pd.ExcelWriter(output_file, engine='xlsxwriter')

    workbook = writer.book
    fmt_jam = workbook.add_format({'num_format': '#,##0.00 "JMD"'}) 
    fmt_usd = workbook.add_format({'num_format': '#,##0.00 "USD"'}) 
    fmt_std = workbook.add_format({'num_format': '#,##0.00'})

    for entidad, reglas in reglas_entidades.items():
        condicion_xq = (df_acc['Journal Entry Type'].str.contains('XQ', na=False, case=False)) & \
                       (df_acc['Journal Entry Item Text'].str.contains('|'.join(reglas['xq_ba']), na=False, regex=True, case=False))
        df_xq = df_acc[condicion_xq].copy()
        

        condicion_bank = (df_acc['Journal Entry Type'].str.contains('ZB|ZR|BR|IB', na=False, case=False)) & \
                         (df_acc['Journal Entry Item Text'].str.contains('|'.join(reglas['bank_text']), na=False, regex=True, case=False))
        df_bank = df_acc[condicion_bank].copy()
        
        
        total_xq = df_xq['Amount in Transaction Currency'].sum()
        total_bank = df_bank['Amount in Transaction Currency'].sum()
        diferencia = total_xq + total_bank
        nom_text = f"{reglas['prefix']} {abs(total_xq):.2f} BK{abs(total_bank):.2f} - {MES_NOMBRE} {ANIO}"
        
        filas_finales = df_xq.to_dict('records')
        filas_finales.append({}) 
        filas_finales.append({'Amount in Transaction Currency': total_xq})
        filas_finales.append({}) 
        
        filas_finales.extend(df_bank.to_dict('records'))
        filas_finales.append({}) 
        filas_finales.append({'Amount in Transaction Currency': total_bank})
        filas_finales.append({}) 
        filas_finales.append({
            'Amount in Transaction Currency': diferencia, 
            'Offsetting Account': nom_text 
        })
        
        df_sheet = pd.DataFrame(filas_finales)

        # Logica para limpiar Fiscal Period y Company Code Currency en totales/vacios
        def is_real_data(row):
            gl_val = str(row.get('G/L Account', ''))
            return pd.notna(row.get('G/L Account')) and gl_val != "" and gl_val.lower() != "nan"

        if 'Amount in Transaction Currency' in df_sheet.columns:
            # Solo poner JMD y 002 si hay una cuenta contable real en esa fila
            df_sheet['Amount in Company Code Currency'] = df_sheet.apply(
                lambda x: x['Amount in Transaction Currency'] if is_real_data(x) else "", 
                axis=1
            )
            df_sheet['Fiscal Period'] = df_sheet.apply(
                lambda x: MES_NUMERO if is_real_data(x) else "", 
                axis=1
            )
        
        # Formatear fechas Dia/Mes/Ano
        for date_col in ['Posting Date', 'Value date', 'Clearing Date']:
            if date_col in df_sheet.columns:
                df_sheet[date_col] = df_sheet[date_col].apply(clean_date)
                
        for col in COLUMNAS_SALIDA:
            if col not in df_sheet.columns: df_sheet[col] = ""
        df_sheet = df_sheet[COLUMNAS_SALIDA]
        
        df_sheet.to_excel(writer, sheet_name=entidad, index=False)
        
        worksheet = writer.sheets[entidad]
        for i, col in enumerate(df_sheet.columns):
            col_len = df_sheet[col].map(lambda x: len(str(x)) if pd.notna(x) else 0).max()
            max_len = max(col_len, len(str(col))) + 1
            
            if col in ['Amount in Transaction Currency', 'Amount in Company Code Currency']:
                worksheet.set_column(i, i, max_len, fmt_jam)
            elif col == 'Amount in Global Currency':
                worksheet.set_column(i, i, max_len, fmt_usd)
            elif 'Amount' in col:
                worksheet.set_column(i, i, max_len, fmt_std)
            else:
                worksheet.set_column(i, i, max_len)

    writer.close()
    if IS_S3:
        output_buffer.seek(0)
        write_excel_to_path(output_buffer, output_file)
    print(f"Reporte Maestro Correcto Generado: {output_file}")

Reporte Maestro Correcto Generado: Automatic reconciliation/11137280/11137280_November_2025.xlsx
Reporte Maestro Correcto Generado: Automatic reconciliation/11137280/11137280_December_2025.xlsx
Reporte Maestro Correcto Generado: Automatic reconciliation/11137280/11137280_January_2026.xlsx
Reporte Maestro Correcto Generado: Automatic reconciliation/11137280/11137280_February_2026.xlsx
